# AlphaZero pour Songo — Entraînement Colab Pro (v2)

**Architecture** : Workers CPU parallèles (self-play) + GPU (training)

Le goulot d'étranglement du MCTS est le parcours d'arbre en Python, pas l'inférence neuronale.
Notre modèle (3.4M params) tourne aussi vite sur CPU que sur GPU.
On exploite donc **tous les CPU cores** pour le self-play en parallèle, et le **GPU pour le training** (opérations batch sur tenseurs).

**Workflow :**
1. Vérifier GPU + CPU cores
2. Monter Google Drive (persistance)
3. Uploader `training.zip`
4. Configurer et lancer
5. Si coupure → `RESUME = True` et relancer

> Les checkpoints + replay buffer se sauvegardent sur Drive toutes les 5 itérations.

In [ ]:
# ── Cellule 1 : Vérifier le GPU + CPU ─────────────────────────────────────────
import torch
import os

cpu_count = os.cpu_count()
print(f"PyTorch version : {torch.__version__}")
print(f"CPU cores       : {cpu_count}")
print(f"GPU disponible  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU             : {gpu_name}")
    print(f"VRAM            : {vram_gb:.1f} GB")
else:
    gpu_name = "CPU"
    print("\n⚠️ Pas de GPU — Runtime → Change runtime type → GPU")

print(f"\n📋 Stratégie d'entraînement :")
print(f"   Self-play : {cpu_count} workers CPU en parallèle")
print(f"   (le bottleneck est Python/MCTS, pas l'inférence NN)")
print(f"   Training  : GPU (opérations batch sur tenseurs)")
print(f"   Pit eval  : GPU (inférence rapide)")

In [ ]:
# ── Cellule 2 : Monter Google Drive ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR = '/content/drive/MyDrive/songo_alphazero'
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'
REPLAY_BUFFER_PATH = f'{CHECKPOINT_DIR}/replay_buffer.pkl'

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Vérifier si un entraînement précédent existe
existing_files = os.listdir(CHECKPOINT_DIR) if os.path.exists(CHECKPOINT_DIR) else []
if existing_files:
    print(f"📂 Checkpoints existants trouvés :")
    for f in sorted(existing_files):
        size_mb = os.path.getsize(f'{CHECKPOINT_DIR}/{f}') / 1e6
        print(f"   {f} ({size_mb:.1f} MB)")
    print(f"\n→ Pour reprendre, mettre RESUME = True dans la cellule config")
else:
    print(f"📂 Aucun checkpoint existant — entraînement from scratch")

print(f"\nDossier Drive : {DRIVE_DIR}")

In [ ]:
# ── Cellule 3 : Uploader le zip training ──────────────────────────────────────
# Sur votre PC, zipper le dossier training/ (sans venv ni tests) :
#   - Windows PowerShell :
#     cd D:\akong-online
#     Compress-Archive -Path training\*.py, training\requirements.txt, training\colab_train.ipynb -DestinationPath training.zip -Force
#
# Puis exécuter cette cellule et uploader le fichier training.zip

from google.colab import files
import zipfile
import os

print("📤 Uploader training.zip ...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"\n📦 Extraction de {zip_name} ...")

WORK_DIR = '/content/training'
os.makedirs(WORK_DIR, exist_ok=True)

# Extraire et détecter la structure du zip
with zipfile.ZipFile(zip_name, 'r') as z:
    names = z.namelist()
    # Vérifier si les fichiers sont dans un sous-dossier training/ ou à la racine
    has_subfolder = any(n.startswith('training/') for n in names)

    if has_subfolder:
        # Fichiers dans training/ → extraire dans /content/
        z.extractall('/content/')
    else:
        # Fichiers à la racine du zip → extraire directement dans /content/training/
        z.extractall(WORK_DIR)

# Vérifier les fichiers essentiels
essential = ['config.py', 'songo_game.py', 'neural_network.py', 'mcts.py',
             'self_play.py', 'train.py', 'pit_evaluation.py', 'export_onnx.py']

print(f"\n✅ Fichiers extraits dans {WORK_DIR} :")
missing = []
for f in essential:
    path = f'{WORK_DIR}/{f}'
    if os.path.exists(path):
        print(f"   ✓ {f}")
    else:
        print(f"   ✗ {f} — MANQUANT !")
        missing.append(f)

if missing:
    print(f"\n❌ Fichiers manquants : {missing}")
else:
    print(f"\n✅ Tous les fichiers essentiels sont présents !")

In [ ]:
# ── Cellule 4 : Installer les dépendances ─────────────────────────────────────
# PyTorch + CUDA est déjà installé sur Colab. On ajoute le reste.
!pip install -q onnx onnxruntime tqdm
print("✅ Dépendances installées")

In [ ]:
# ── Cellule 5 : Configuration ──────────────────────────────────────────────────
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Pourquoi 400 sims et pas 1600 ?                                     ║
# ║  - Songo a 7 actions possibles, Go en a 361                          ║
# ║  - AlphaGo Zero: 1600 sims pour 361 actions = 4.4 sims/action       ║
# ║  - Songo:        400 sims pour 7 actions   = 57 sims/action          ║
# ║  - On explore MIEUX l'arbre avec 400 sims au Songo qu'AlphaGo !     ║
# ║                                                                       ║
# ║  Le vrai levier: nombre de PARTIES, pas sims/coup                    ║
# ╚═══════════════════════════════════════════════════════════════════════╝

import os

NUM_ITERATIONS   = 60    # Itérations totales
GAMES_PER_ITER   = 100   # Parties self-play par itération
MCTS_SIMS        = 400   # 400 sims = 57 sims/action (très bon pour 7 actions)
MCTS_BATCH       = 8     # Batch inference CPU (8 suffisant)
TRAIN_BATCH      = 256   # Batch entraînement NN (GPU)
TRAIN_EPOCHS     = 10    # Époques de training par itération
LEARNING_RATE    = 0.001 # LR initiale
NUM_WORKERS      = 0     # 0 = auto-detect CPU cores

# Reprise d'un entraînement précédent
RESUME = False  # Mettre True si des checkpoints existent sur Drive

# ─── Résumé ──────────────────────────────────────────────────────────────
workers = NUM_WORKERS or os.cpu_count()
total_games = NUM_ITERATIONS * GAMES_PER_ITER

print("=" * 55)
print("  AlphaZero v2 — Config Colab Pro")
print("=" * 55)
print(f"  Itérations       : {NUM_ITERATIONS}")
print(f"  Games/itération  : {GAMES_PER_ITER}")
print(f"  Total parties    : {total_games} (x2 avec symétrie miroir)")
print(f"  MCTS sims/coup   : {MCTS_SIMS} ({MCTS_SIMS/7:.0f} sims/action)")
print(f"  Workers CPU      : {workers} (parallèle)")
print(f"  Train batch      : {TRAIN_BATCH} (GPU)")
print(f"  LR               : {LEARNING_RATE}")
print(f"  Checkpoints      : {CHECKPOINT_DIR}")
print(f"  Reprise          : {'OUI' if RESUME else 'NON (from scratch)'}")
print("=" * 55)

# Estimation temps
# 400 sims × ~100 coups/partie ≈ 20-30s/partie sur CPU
# 100 parties / N workers ≈ games_per_worker parties chacun
games_per_worker = GAMES_PER_ITER / workers
est_sp_min = games_per_worker * 25 / 60  # ~25s/partie sur CPU
est_train_min = 1  # training + pit ~1-2 min
est_iter_min = est_sp_min + est_train_min
est_total_h = est_iter_min * NUM_ITERATIONS / 60

print(f"\n⏱️  ~{est_iter_min:.0f} min/itération ({est_sp_min:.0f} self-play + {est_train_min} train/pit)")
print(f"⏱️  ~{est_total_h:.1f}h total estimé")

# Coût Colab
if torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0):
    units = est_total_h * 7
    print(f"💰  ~{units:.0f} unités Colab (A100 @ ~7 unités/h)")
    print(f"   Reste ~{100 - units:.0f} unités pour une session de reprise")

In [ ]:
# ── Cellule 6 : Lancer l'entraînement ─────────────────────────────────────────
import sys
import os

# Ajouter le dossier training au path
WORK_DIR = '/content/training'
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

from config import AlphaZeroConfig
from train import train

# ── Construire la config ──────────────────────────────────────────────────
config = AlphaZeroConfig()

# Self-play
config.training.num_iterations      = NUM_ITERATIONS
config.training.num_self_play_games = GAMES_PER_ITER
config.mcts.num_simulations         = MCTS_SIMS
config.training.mcts_batch_size     = MCTS_BATCH

# Parallélisme : workers CPU pour self-play, GPU pour training
config.training.num_workers         = NUM_WORKERS  # 0 = auto-detect

# Training
config.training.batch_size          = TRAIN_BATCH
config.training.num_epochs          = TRAIN_EPOCHS
config.training.learning_rate       = LEARNING_RATE

# Checkpoints → Google Drive (persistent)
config.training.checkpoint_dir      = CHECKPOINT_DIR

# ── Reprise ou from scratch ──────────────────────────────────────────────
resume_path = None
if RESUME:
    latest = f'{CHECKPOINT_DIR}/model_latest.pt'
    champion = f'{CHECKPOINT_DIR}/model_champion.pt'
    if os.path.exists(latest):
        resume_path = latest
        print(f"🔄 Reprise depuis : {latest}")
    elif os.path.exists(champion):
        resume_path = champion
        print(f"🔄 Reprise depuis : {champion}")
    else:
        print("⚠️ RESUME=True mais aucun checkpoint trouvé → from scratch")

print(f"\n🚀 Lancement — {NUM_WORKERS or os.cpu_count()} workers CPU + GPU training\n")
model = train(config, resume_from=resume_path)

In [ ]:
# ── Cellule 7 : Exporter le champion en ONNX ──────────────────────────────────
# Exécuter cette cellule APRÈS l'entraînement pour récupérer le modèle web
import sys
sys.path.insert(0, '/content/training')
os.chdir('/content/training')

from export_onnx import export_to_onnx
import onnx

champion_path = f'{CHECKPOINT_DIR}/model_champion.pt'
onnx_path = f'{DRIVE_DIR}/songo_nn.onnx'

if os.path.exists(champion_path):
    # Export brut
    tmp_onnx = '/content/songo_nn_tmp.onnx'
    export_to_onnx(champion_path, tmp_onnx, quantize=False)

    # Pack en un seul fichier (pour ONNX Runtime Web)
    m = onnx.load(tmp_onnx)
    onnx.save_model(m, onnx_path, save_as_external_data=False)

    size_mb = os.path.getsize(onnx_path) / 1e6
    print(f"\n✅ ONNX exporté : {onnx_path} ({size_mb:.1f} MB)")
    print(f"   → Télécharger et placer dans public/songo_nn.onnx du projet web")

    # Cleanup
    if os.path.exists(tmp_onnx):
        os.remove(tmp_onnx)
    if os.path.exists(tmp_onnx + '.data'):
        os.remove(tmp_onnx + '.data')
else:
    print(f"❌ Champion introuvable : {champion_path}")
    print("   L'entraînement doit d'abord produire au moins une promotion.")

In [ ]:
# ── Cellule 8 : Télécharger le modèle ONNX ────────────────────────────────────
from google.colab import files

if os.path.exists(onnx_path):
    files.download(onnx_path)
    print("📥 Téléchargement lancé — placer dans public/songo_nn.onnx")
else:
    print("❌ Fichier ONNX non trouvé. Exécuter la cellule 7 d'abord.")

---

## 🔄 Reprendre un entraînement interrompu

Si la session Colab a été coupée :

1. Relancer les **cellules 1 à 4** (GPU, Drive, Upload zip, Install)
2. Dans la **cellule 5**, mettre `RESUME = True`
3. Relancer la **cellule 6** — l'entraînement reprend au dernier checkpoint

Les checkpoints + replay buffer sont sur Google Drive, rien n'est perdu.

## 📊 Surveiller la progression

Pendant l'entraînement, surveillez :
- **Policy loss** qui descend (< 1.0 = bon, < 0.5 = excellent)
- **Value loss** qui descend (< 0.3 = bon)
- **Promotions** : chaque promotion = le modèle s'améliore
- **Échecs consécutifs** : 3 d'affilée → reset automatique aux poids champion

## 📁 Fichiers sur Google Drive

```
MyDrive/songo_alphazero/
├── checkpoints/
│   ├── model_champion.pt      ← Meilleur modèle connu
│   ├── model_champion_prev.pt ← Backup du champion précédent
│   ├── model_latest.pt        ← Dernier état (pour reprendre)
│   └── replay_buffer.pkl      ← Buffer de données (persistent)
└── songo_nn.onnx              ← Export pour le web (après cellule 7)
```